In [15]:
import os
import json
from dotenv import load_dotenv
from IPython.display import Markdown, display, update_display
from new_scraper import fetch_website_links, fetch_website_contents
from openai import OpenAI

In [16]:
links = fetch_website_links("https://edwarddonner.com")

In [17]:
links

['#wp--skip-link--target',
 'https://edwarddonner.com/avatar/',
 'https://edwarddonner.com/curriculum/',
 'https://edwarddonner.com/proficient/',
 'https://edwarddonner.com/connect-four/',
 'https://edwarddonner.com/outsmart/',
 'https://edwarddonner.com/about-me-and-about-nebula/',
 'https://edwarddonner.com/posts/',
 'https://edwarddonner.com/',
 'https://news.ycombinator.com',
 'https://nebula.io/?utm_source=ed&utm_medium=referral',
 'https://www.prnewswire.com/news-releases/wynden-stark-group-acquires-nyc-venture-backed-tech-startup-untapt-301269512.html',
 'https://edwarddonner.com/curriculum/',
 'https://edwarddonner.com/avatar/',
 'https://edwarddonner.com/2026/02/17/ai-coder-vibe-coder-to-agentic-engineer/',
 'https://edwarddonner.com/2026/02/17/ai-coder-vibe-coder-to-agentic-engineer/',
 'https://edwarddonner.com/2026/01/04/ai-builder-with-n8n-create-agents-and-voice-agents/',
 'https://edwarddonner.com/2026/01/04/ai-builder-with-n8n-create-agents-and-voice-agents/',
 'https:/

## Step 1 - llama figures out the relevant links

In [18]:
link_system_prompt = """
You are provided with a list of links found on a webpage.
You are able to decide which of the links would be most relevant to include in a brochure about the company,
such as links to an About page, or a Company page, or Careers/Jobs pages.
You should respond in JSON as in this example:

{
    "links": [
        {"type": "about page", "url": "https://full.url/goes/here/about"},
        {"type": "careers page", "url": "https://another.full.url/careers"}
    ]
}
"""

In [19]:
def get_links_user_prompt(url):
    user_prompt = f"""
Here is the list of links on the website {url} -
Please decide which of these are relevant web links for a brochure about the company, 
respond with the full https URL in JSON format.
Do not include Terms of Service, Privacy, email links.

Links (some might be relative links):

"""
    links = fetch_website_links(url)
    user_prompt += "\n".join(links)
    return user_prompt

In [20]:
print(get_links_user_prompt("https://edwarddonner.com"))


Here is the list of links on the website https://edwarddonner.com -
Please decide which of these are relevant web links for a brochure about the company, 
respond with the full https URL in JSON format.
Do not include Terms of Service, Privacy, email links.

Links (some might be relative links):

#wp--skip-link--target
https://edwarddonner.com/avatar/
https://edwarddonner.com/curriculum/
https://edwarddonner.com/proficient/
https://edwarddonner.com/connect-four/
https://edwarddonner.com/outsmart/
https://edwarddonner.com/about-me-and-about-nebula/
https://edwarddonner.com/posts/
https://edwarddonner.com/
https://news.ycombinator.com
https://nebula.io/?utm_source=ed&utm_medium=referral
https://www.prnewswire.com/news-releases/wynden-stark-group-acquires-nyc-venture-backed-tech-startup-untapt-301269512.html
https://edwarddonner.com/curriculum/
https://edwarddonner.com/avatar/
https://edwarddonner.com/2026/02/17/ai-coder-vibe-coder-to-agentic-engineer/
https://edwarddonner.com/2026/02/17

### Setup our free ollama model lol

In [21]:
OLLAMA_BASE_URL = "http://localhost:11434/v1"
from openai import OpenAI
ollama = OpenAI(base_url=OLLAMA_BASE_URL, api_key='ollama')

In [22]:
def select_relevant_links(url):
    response = ollama.chat.completions.create(
        model="llama3.2",
        messages=[
            {"role": "system", "content": link_system_prompt},
            {"role": "user", "content": get_links_user_prompt(url)}
        ],
        response_format={"type": "json_object"}
    )
    result = response.choices[0].message.content
    links = json.loads(result)
    return links
    

In [24]:
select_relevant_links("https://edwarddonner.com")

{'links': [{'type': 'about page',
   'url': 'https://edwarddonner.com/about-me-and-about-nebula/'},
  {'type': 'home page', 'url': 'https://edwarddonner.com/'},
  {'type': 'blog posts', 'url': 'https://edwarddonner.com/posts/'},
  {'type': 'linkedin profile', 'url': 'https://www.linkedin.com/in/eddonner/'},
  {'type': 'twitter profile', 'url': 'https://twitter.com/edwarddonner'},
  {'type': 'facebook profile',
   'url': 'https://www.facebook.com/edward.donner.52'},
  {'type': 'Company page',
   'url': 'https://nebula.io/?utm_source=ed&utm_medium=referral'},
  {'type': 'News releases',
   'url': 'https://www.prnewswire.com/news-releases/wynden-stark-group-acquires-nyc-venture-backed-tech-startup-untapt-301269512.html'}]}

In [25]:
def select_relevant_links(url):
    print(f"Selecting relevant links for {url} by calling llama3.2")
    response = ollama.chat.completions.create(
        model="llama3.2",
        messages=[
            {"role": "system", "content": link_system_prompt},
            {"role": "user", "content": get_links_user_prompt(url)}
        ],
        response_format={"type": "json_object"}
    )
    result = response.choices[0].message.content
    links = json.loads(result)
    print(f"Found {len(links['links'])} relevant links")
    return links

In [26]:
select_relevant_links("https://edwarddonner.com")

Selecting relevant links for https://edwarddonner.com by calling llama3.2
Found 4 relevant links


{'links': [{'type': 'about page',
   'url': 'https://edwarddonner.com/about-me-and-about-nebula'},
  {'type': 'home page or Company page', 'url': 'https://edwarddonner.com'},
  {'type': 'LinkedIn profile', 'url': 'https://www.linkedin.com/in/eddonner/'},
  {'type': 'Twitter handle', 'url': 'https://twitter.com/edwarddonner'}]}

## Step 2 - Make the brochure

In [31]:
from bs4 import BeautifulSoup
import requests
from urllib.parse import urlparse, urljoin

In [30]:
def normalize_url(url, base_url=None):
    """
    Clean and normalize URLs before fetching.

    - Converts relative URLs to absolute URLs
    - Ignores mailto/tel/javascript links
    - Fixes suspicious subdomains like www.about.udemy.com -> about.udemy.com
    - Removes URL fragments like #section
    """

    if not url:
        return None

    url = url.strip()

    # Ignore non-web links
    if url.startswith(("mailto:", "tel:", "javascript:", "#")):
        return None

    # Convert relative URL to absolute URL
    if base_url:
        url = urljoin(base_url, url)

    parsed = urlparse(url)

    # Ignore if there is no valid scheme/domain
    if parsed.scheme not in ("http", "https") or not parsed.netloc:
        return None

    netloc = parsed.netloc.lower()

    # Fix cases like:
    # www.about.udemy.com -> about.udemy.com
    if netloc.startswith("www.about."):
        netloc = netloc.replace("www.about.", "about.", 1)

    # Remove fragment part from URLs
    parsed = parsed._replace(netloc=netloc, fragment="")

    return parsed.geturl()

In [32]:
def fetch_page_and_all_relevant_links(url):
    contents = fetch_website_contents(url)
    relevant_links = select_relevant_links(url)

    result = f"## Landing Page:\n\n{contents}\n## Relevant Links:\n"

    for link in relevant_links["links"]:
        fixed_url = normalize_url(link["url"])

        result += f"\n\n### Link: {link['type']}\n"
        result += f"\nURL: {fixed_url}\n"

        if fixed_url:
            result += fetch_website_contents(fixed_url)
        else:
            result += "\nInvalid or unsupported URL skipped."

    return result

In [33]:
print(fetch_page_and_all_relevant_links("https://www.apnacollege.in"))

Selecting relevant links for https://www.apnacollege.in by calling llama3.2
Found 7 relevant links
## Landing Page:

Home Main

New Sigma 12
New Courses
NEW Sigma 12 (Complete Placement Prep)
Delta (Complete Web Development)
Sigma 11 + AI/ML
Prime 2.0 (Complete AI/ML Batch)
Alpha Plus (Complete DSA)
Our Results
DSA Sheet
Log in
Learn & become the
Top 1% software developer
Alpha Plus - DSA (Java/C++) | Delta - Web Development | Sigma - DSA + Web Development + Quant-Aptitude
Ultimate
Placement Solution
India's Most Loved Coding Community ❤️
Happy Learners
monthly views
new monthly subscribers
New PLACEMENT PREP Batch🔥
Sigma 12 : Development + DSA + Aptitude
Start your placement preparation today!
Explore more
Thousands of students achieved their
dream job at
+ many more
Our Selected Students
See more
On a mission to teach Millions
Join us on
#youtube
|
#instagram
|
#telegram
Youtube
Apna College
Instagram
aman DHATTARWAL
Telegram
APNA COLLEGE
Where education meets real-world needs.
Helpf

In [34]:
brochure_system_prompt = """
You are an assistant that analyzes the contents of several relevant pages from a company website
and creates a short brochure about the company for prospective customers, investors and recruits.
Respond in markdown without code blocks.
Include details of company culture, customers and careers/jobs if you have the information.
"""

# a more humorous brochure

# brochure_system_prompt = """
# You are an assistant that analyzes the contents of several relevant pages from a company website
# and creates a short, humorous, entertaining, witty brochure about the company for prospective customers, investors and recruits.
# Respond in markdown without code blocks.
# Include details of company culture, customers and careers/jobs if you have the information.
# """

In [35]:
def get_brochure_user_prompt(company_name, url):
    user_prompt = f"""
You are looking at a company called: {company_name}
Here are the contents of its landing page and other relevant pages;
use this information to build a short brochure of the company in markdown without code blocks.\n\n
"""
    user_prompt += fetch_page_and_all_relevant_links(url)
    user_prompt = user_prompt[:5_000] # Truncate if more than 5,000 characters
    return user_prompt

In [36]:
get_brochure_user_prompt("apnacollege", "https://www.apnacollege.in")

Selecting relevant links for https://www.apnacollege.in by calling llama3.2
Found 5 relevant links


"\nYou are looking at a company called: apnacollege\nHere are the contents of its landing page and other relevant pages;\nuse this information to build a short brochure of the company in markdown without code blocks.\n\n\n## Landing Page:\n\nHome Main\n\nNew Sigma 12\nNew Courses\nNEW Sigma 12 (Complete Placement Prep)\nDelta (Complete Web Development)\nSigma 11 + AI/ML\nPrime 2.0 (Complete AI/ML Batch)\nAlpha Plus (Complete DSA)\nOur Results\nDSA Sheet\nLog in\nLearn & become the\nTop 1% software developer\nAlpha Plus - DSA (Java/C++) | Delta - Web Development | Sigma - DSA + Web Development + Quant-Aptitude\nUltimate\nPlacement Solution\nIndia's Most Loved Coding Community ❤️\nHappy Learners\nmonthly views\nnew monthly subscribers\nNew PLACEMENT PREP Batch🔥\nSigma 12 : Development + DSA + Aptitude\nStart your placement preparation today!\nExplore more\nThousands of students achieved their\ndream job at\n+ many more\nOur Selected Students\nSee more\nOn a mission to teach Millions\nJoi

In [37]:
def stream_brochure(company_name, url):
    stream = ollama.chat.completions.create(
        model="llama3.2",
        messages=[
            {"role": "system", "content": brochure_system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
          ],
        stream=True
    )    
    response = ""
    display_handle = display(Markdown(""), display_id=True)
    for chunk in stream:
        response += chunk.choices[0].delta.content or ''
        update_display(Markdown(response), display_id=display_handle.display_id)

In [43]:
stream_brochure("apnacollege", "https://www.apnacollege.in")

Selecting relevant links for https://www.apnacollege.in by calling llama3.2
Found 7 relevant links


**Welcome to Apna College**

Empowering Tomorrow's Tech Leaders Today

At Apna College, we are on a mission to teach millions of students across the world real-world skills in technology. Our innovative approach blends classroom training with placement prep, ensuring that our students achieve their dream jobs.

**Our Mission**

We believe that education is not just about acquiring knowledge, but about equipping individuals with practical skills to succeed in the industry. At Apna College, we strive to make technology accessible and useful for everyone, regardless of their background or location.

**How We Work**

* Providing top-notch courses, including programming languages, data structures, algorithms, machine learning, and more.
* Offering comprehensive placement prep programs that cover mock interviews, coding challenges, and resume building.
* Creating a supportive community where students can connect with like-minded individuals, learn from each other's experiences, and grow together.

**What Our Students Say**

Thousands of our students have achieved great success in the tech industry, securing placements at top companies like PayPal, Blinkit, Goldman Sachs, Oracle, Zomato, Amdocs, and more. They credited their success to Apna College's rigorous training program and supportive community.

**Meet Our Founder & CEO**

Aman Dhattarwal is the visionary founder and CEO of Apna College. With a passion for education and technology, he has created a platform that has connected over 9 million students across India and worldwide. Follow us on social media to learn more about our journeys and achievements.

**Stay Connected with Us**

Get in touch:
 Phone: +91-120-4064444
 Email: [info@apnacollege.in](mailto:info@apnacollege.in)
 LinkedIn: linkedin.com/company/apnacollege
 Facebook: facebook.com/dhattarwalaman

**Join Our Community**

Stay updated with the latest news, success stories, and job opportunities by following us on social media or subscribing to our newsletter. Let's connect, learn, and grow together!